<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-07-blob-processing-classification-routing/notebooks/Ophelie_Full_Segmentation_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the Full Segmentation Pipeline with Metadata Output

Runs the full document-segmentation pipeline over `pharma-blob-sample.pdf` (10 pages) using an
**open-source model only**. No Gemini anywhere in this notebook.

**Model:** Mistral-7B-Instruct-v0.2, Q4_K_M GGUF, via llama.cpp.

Chosen on the strength of the evaluation in the previous module, where Mistral matched Gemini on
fact recall (14/15 against 14/15) and was the only local model that did not fabricate anything.
Phi-2 scored 5/15 and denied a fact that was in its own prompt; TinyLlama scored 9/15 and
contradicted itself. For regulated documents, not inventing things matters more than speed.

**One fix carried over from that task.** Last time llama-cpp-python was installed from the prebuilt
cu123 wheel index and silently fell back to a CPU-only build, so Mistral generated at 0.18 to 0.41
tokens/sec even with `n_gpu_layers=-1`. Colab now runs Python 3.13, which those prebuilt wheels do
not cover. This notebook compiles llama-cpp-python from source with `GGML_CUDA=on` instead, and
verifies that the layers actually reach the GPU before running anything.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. No API key is needed.

## Step 1: Build llama.cpp with CUDA

This compiles from source and takes roughly 8 to 15 minutes. That is the price of getting the GPU to
actually engage; the alternative is a 40-minute pipeline run on CPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nvcc --version | tail -2

name, memory.total [MiB]
Tesla T4, 15360 MiB
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%time
!pip install -q PyMuPDF gdown
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q llama-cpp-python --no-cache-dir --force-reinstall --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 154.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 267.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 309.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 286.6 MB/s eta 0:00:00


## Step 2: Extract page-level text with PyMuPDF

In [ ]:
import os, fitz

PDF_PATH = "/content/pharma-blob-sample.pdf"
DRIVE_FILE_ID = "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"

if not os.path.exists(PDF_PATH):
    !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O {PDF_PATH} || true

doc = fitz.open(PDF_PATH)
doc_pages = [{"page_num": i, "text": page.get_text()} for i, page in enumerate(doc)]

print(f"{len(doc_pages)} pages extracted\n")
for p in doc_pages:
    head = (p["text"] or "").strip().split("\n")[0][:60]
    print(f"page {p['page_num']:>2} | {len(p['text'] or ''):>5} chars | {head}")

## Step 3: Load Mistral and confirm the GPU is actually being used

`verbose=True` is deliberate. The line to look for is `offloaded 33/33 layers to GPU`. If it says
`offloaded 0/33`, the CUDA build did not take and everything below will run at CPU speed.

In [ ]:
import time
from llama_cpp import Llama

MODEL_PATH = "/content/mistral.gguf"
MODEL_URL = ("https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/"
             "resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf")

if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress {MODEL_URL} -O {MODEL_PATH}
print(f"model file: {os.path.getsize(MODEL_PATH) / (1024**3):.2f} GB")

t0 = time.time()
llm = Llama(model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=4096, verbose=True)
print(f"\nloaded in {time.time() - t0:.1f}s")

In [ ]:
# Throughput probe. On a T4 with real GPU offload this should be tens of tokens/sec.
# The same model on CPU managed 0.18 to 0.41 tok/s in the previous module.
t0 = time.time()
probe = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Count from 1 to 40, numbers separated by spaces."}],
    max_tokens=120, temperature=0.0,
)
dt = time.time() - t0
n = probe["usage"]["completion_tokens"]
print(f"{n} tokens in {dt:.2f}s = {n / dt:.1f} tok/s")
if n / dt < 5:
    print("WARNING: this is CPU-speed. Check the offload line above before trusting the timings.")

## Step 4: The segmentation prompt

One call per page returns both decisions at once, matching the output shape the task specifies:

```
{"is_new_doc": "Yes", "doc_type": "Certificate of Quality"}
```

The prompt names the concrete boundary signals rather than asking a bare "same document?" question,
and it says explicitly that two pages can share a template and still be different documents. That
last line is what separates the two Certificates of Quality in this file, which sit on the same
template and differ only by lot number.

In [ ]:
import json, re

DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other",
]

def build_prompt(prev_text, curr_text, prev_type):
    return f"""You are segmenting a bundled pharmaceutical PDF into separate documents.

PREVIOUS PAGE (already classified as: {prev_type or 'none, this is the first page'}):
{(prev_text or '(none)')[:1500]}

CURRENT PAGE:
{(curr_text or '')[:1500]}

Decide whether the CURRENT page STARTS A NEW document or CONTINUES the previous one.

A NEW document starts when the current page shows:
- a new title or heading block at the top
- a different document number, certificate number, lot number or article number
- a "Page 1 of N" marker

The SAME document continues when the current page shows:
- the word "continued", or a "Page 2 of N" marker
- a numbered section carrying on from the previous page

Two pages can use the SAME template and have the SAME document type and still be TWO DIFFERENT
documents. Compare lot numbers and document numbers before deciding.

If it starts a new document, classify it as exactly one of:
{', '.join(DOC_TYPES)}

If it continues the previous document, repeat the previous document's type.

Respond with ONLY this JSON object and nothing else:
{{"is_new_doc": "Yes or No", "doc_type": "one of the types listed above"}}"""


def ask_model(prompt, max_tokens=60):
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return out["choices"][0]["message"]["content"]

## Step 5: Parsing with fallback handling

Three layers, because a 7B model asked for JSON will not always return clean JSON:

1. Parse the first `{...}` block as JSON.
2. If that fails, pull `is_new_doc` and `doc_type` out of the raw text with regex.
3. If that fails too, treat the page as a continuation of the current document and flag it. Assuming
   continuation is the safer default: a missed split leaves pages joined, which a human reviewer can
   see, whereas a false split invents a document that was never in the file.

The document type is always validated against the fixed list rather than taken as free text.

In [ ]:
def normalise_type(raw, fallback="Other"):
    if not raw:
        return fallback
    raw = raw.strip().strip(".").strip()
    for t in DOC_TYPES:
        if raw.lower() == t.lower():
            return t
    for t in DOC_TYPES:
        if t.lower() in raw.lower():
            return t
    return fallback


def parse_reply(reply, prev_type):
    """Returns (is_new_doc, doc_type, how_it_was_parsed)."""
    # A new document with an unrecognised type falls back to "Other", NOT to the previous
    # document's type. Inheriting the old label would quietly mislabel a genuinely new document.
    def fb(is_new):
        return "Other" if is_new else (prev_type or "Other")

    # Layer 1: real JSON
    m = re.search(r"\{.*?\}", reply, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            is_new = str(obj.get("is_new_doc", "")).strip().lower().startswith("y")
            return is_new, normalise_type(obj.get("doc_type"), fb(is_new)), "json"
        except json.JSONDecodeError:
            pass

    # Layer 2: regex over the raw text
    new_m = re.search(r"is_new_doc\W+(yes|no)", reply, re.I)
    type_m = re.search(r"doc_type\W+\"?([A-Za-z/ ]+)", reply)
    if new_m:
        is_new = new_m.group(1).lower() == "yes"
        return is_new, normalise_type(type_m.group(1) if type_m else None, fb(is_new)), "regex"

    # Layer 3: give up safely, and say so
    return False, (prev_type or "Other"), "FALLBACK-continuation"


results = []
diagnostics = []
current_type = None
page_in_doc = 0
doc_id = 0

for i, page in enumerate(doc_pages):
    text = page["text"] or ""

    if not text.strip():
        # An empty page cannot be judged from text. Mark it and move on rather than guessing.
        is_new, dtype, how = False, (current_type or "Other"), "EMPTY-PAGE (needs OCR)"
        reply = ""
    elif i == 0:
        prompt = build_prompt(None, text, None)
        t0 = time.time()
        reply = ask_model(prompt)
        _, dtype, how = parse_reply(reply, None)
        is_new = True  # the first page always starts a document
    else:
        prompt = build_prompt(doc_pages[i - 1]["text"], text, current_type)
        t0 = time.time()
        reply = ask_model(prompt)
        is_new, dtype, how = parse_reply(reply, current_type)

    if is_new:
        current_type = dtype
        page_in_doc = 0
        if i > 0:
            doc_id += 1
    else:
        dtype = current_type or dtype
        page_in_doc = 0 if i == 0 else page_in_doc + 1

    results.append({
        "page": i,
        "is_new_doc": "Yes" if is_new else "No",
        "doc_type": dtype,
        "page_in_doc": page_in_doc,
    })
    diagnostics.append({
        "page": i,
        "doc_id": doc_id,
        "parsed_via": how,
        "raw_reply": reply.strip().replace("\n", " ")[:120],
    })
    print(f"page {i}: is_new_doc={results[-1]['is_new_doc']:>3} | "
          f"{dtype:<24} | page_in_doc={page_in_doc} | parsed via {how}")

print(f"\n{doc_id + 1} documents detected across {len(results)} pages")

## Step 6: Final metadata output

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
display(df)

print("\nParse diagnostics (how each page's reply was read):")
display(pd.DataFrame(diagnostics))

final_json = json.dumps(results, indent=2)
with open("/content/segmentation_metadata.json", "w") as f:
    f.write(final_json)

print("=" * 78)
print("FINAL JSON")
print("=" * 78)
print(final_json)
print("=" * 78)
print("END OF JSON")

## Step 7: Check against the verified page structure

The true structure of this file was established in the previous task by reading the first line of
every extracted page, so it can be used to score this run rather than eyeballing it.

In [ ]:
VERIFIED = [
    (0, "Yes", "Cover Letter", 0),
    (1, "Yes", "Certificate of Quality", 0),
    (2, "Yes", "Certificate of Quality", 0),
    (3, "Yes", "Packaging Specification", 0),
    (4, "No",  "Packaging Specification", 1),
    (5, "Yes", "BSE/TSE Declaration", 0),
    (6, "Yes", "Material Description", 0),
    (7, "Yes", "Supplier Qualification", 0),
    (8, "No",  "Supplier Qualification", 1),
    (9, "Yes", "Chain of Custody", 0),
]

rows = []
for (pg, exp_new, exp_type, exp_idx), got in zip(VERIFIED, results):
    rows.append({
        "page": pg,
        "expected_is_new": exp_new, "got_is_new": got["is_new_doc"],
        "boundary_ok": exp_new == got["is_new_doc"],
        "expected_type": exp_type, "got_type": got["doc_type"],
        "type_ok": exp_type == got["doc_type"],
        "expected_page_in_doc": exp_idx, "got_page_in_doc": got["page_in_doc"],
        "index_ok": exp_idx == got["page_in_doc"],
    })

check = pd.DataFrame(rows)
display(check)
print(f"boundaries correct : {check.boundary_ok.sum()}/10")
print(f"types correct      : {check.type_ok.sum()}/10")
print(f"page_in_doc correct: {check.index_ok.sum()}/10")